# 10 — Figure 2/3 PySR seed stability

Repeat Notebook 09's covariance-aware PySR fit on the fixed Notebook 08 datasets, changing only `random_state`. Figure 2 uses DESI + Pantheon+; Figure 3 uses DESI + DES-SN5YR. The public DESI errors are primary. The rounded Table I error construction is a sensitivity comparison, not the authors' original covariance.

This measures **search stability conditional on fixed data**. Between-seed variation is not measurement uncertainty and is not a $w(z)$ confidence interval. Notebook 11 handles uncertainty propagation.

## 1. Load the Notebook 09 settings and Notebook 08 data

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'configs/pysr_fig23.yaml').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loading import load_standard_dataset, with_desi_error_version
from src.symbolic_regression import load_pysr_config
from src.seed_stability import (setup_experiment, run_seed_ensemble, save_progress,
                                summarize_runs, save_summary_plots)

CONFIG = deepcopy(load_pysr_config(PROJECT_ROOT / 'configs/pysr_fig23.yaml'))
CONFIG['output']['root'] = 'results/10_fig23_seed_stability'
SCENARIOS = ('fig2_pantheon', 'fig3_desy5')
PREPARED_PATHS = {name: PROJECT_ROOT / 'results/08_fig23_data_preparation' / f'{name}.npz'
                  for name in SCENARIOS}
BASE_DATA = {name: load_standard_dataset(path) for name, path in PREPARED_PATHS.items()}
PAPER_EXPRESSIONS = {'fig2_pantheon': '33.3*z*exp(-0.274*z)',
                     'fig3_desy5': '30.1*z*exp(-0.262*z)'}
print('Dataset sizes:', {name: len(data['y']) for name, data in BASE_DATA.items()})
print('Fixed search settings:', CONFIG['search'])


## 2. Define the scenario × covariance × seed experiment

The default plan is 100 seeds per variant. Change `N_SEEDS` to 500 later only if justified; an existing experiment can be extended without repeating completed seeds. `RUN_FITS=False` prevents **Run All** from accidentally launching hundreds of PySR jobs. Set it to `True` when ready. Rerunning this cell in the same kernel reuses its experiment folder; after a kernel restart, put the printed path in `RESUME_DIRECTORY`. Set `RESUME_DIRECTORY=None` explicitly to start a new experiment. Keep all settings and the original seed prefix unchanged when resuming.

In [ ]:
N_SEEDS = 100
SEED_START = 20261004
SEEDS = list(range(SEED_START, SEED_START + N_SEEDS))
ERROR_VERSIONS = tuple(CONFIG['reproducibility']['error_versions'])
RUN_FITS = False  # Change to True to run or resume the ensemble.
RESUME_DIRECTORY = globals().get('EXPERIMENT_DIR')  # Or set an existing path; use None for a new experiment.

VARIANTS = {(name, version): with_desi_error_version(BASE_DATA[name], version)
            for name in SCENARIOS for version in ERROR_VERSIONS}
for name in SCENARIOS:
    if 'paper_table' in ERROR_VERSIONS:
        public, paper = VARIANTS[name, 'public_release'], VARIANTS[name, 'paper_table']
        np.testing.assert_array_equal(public['z'], paper['z'])
        np.testing.assert_array_equal(public['y'], paper['y'])
        np.testing.assert_array_equal(public['covariance'][6:, 6:], paper['covariance'][6:, 6:])
EXPERIMENT_DIR = setup_experiment(
    PROJECT_ROOT / CONFIG['output']['root'], CONFIG, PREPARED_PATHS,
    SCENARIOS, ERROR_VERSIONS, SEEDS, PAPER_EXPRESSIONS,
    resume_directory=RESUME_DIRECTORY)
print('Experiment folder:', EXPERIMENT_DIR)
print('Planned fits:', len(VARIANTS) * N_SEEDS)
display(pd.DataFrame([{'scenario': name, 'error_version': version,
                       'n_data': len(data['y']), 'n_seeds': N_SEEDS}
                      for (name, version), data in VARIANTS.items()]))


## 3. Run or resume the seed ensemble

A completed seed is saved immediately, with its selected expression, full hall of fame, whitening matrix, distance grid, metadata, and a `run_complete.json` marker. Completed seeds are skipped on resume. A failed or interrupted attempt remains on disk; the next run creates a fresh attempt for that seed.

In [ ]:
if RUN_FITS:
    MANIFEST, SELECTED = run_seed_ensemble(EXPERIMENT_DIR, VARIANTS, CONFIG, PAPER_EXPRESSIONS)
else:
    MANIFEST, SELECTED = save_progress(EXPERIMENT_DIR)
    print('Dry run only: set RUN_FITS = True in section 2 to launch fits.')
display(MANIFEST.groupby(['scenario', 'desi_error_version', 'status'])
        .size().rename('count').reset_index())


## 4. Classify the selected equations

Classification is descriptive, not a proof of physical equivalence. Exact $az\exp(bz)$ expressions receive comparable $a,b$ values; other expressions receive broad structural labels. The summary uses only the `best`-selected expression from each seed. All other candidates remain in each fit's hall of fame.

In [ ]:
if SELECTED.empty:
    print('No completed fits yet.')
else:
    display(SELECTED[['scenario', 'desi_error_version', 'seed', 'family',
                      'exp_a', 'exp_b', 'python_loss', 'sympy_expression']].head(20))
    display(SELECTED.groupby(['scenario', 'desi_error_version', 'family'])
            .size().rename('count').reset_index())


## 5. Compare Figure 2 and Figure 3 stability

Structure fractions use completed seeds as their denominator; completion fractions use all planned seeds. Physical checks are numerical checks on the configured redshift grid, not analytical proofs. Curve quantiles describe between-seed variation of **central fits**, not observational confidence bands. Non-finite predictions are omitted pointwise and counted in `n_finite`.

In [ ]:
if SELECTED.empty:
    print('No completed fits to summarize yet.')
else:
    STABILITY, FAMILIES, CURVES, SELECTED = summarize_runs(EXPERIMENT_DIR)
    display(STABILITY)
    display(FAMILIES)
    LOSS_PLOT, DISTANCE_PLOT = save_summary_plots(EXPERIMENT_DIR, PAPER_EXPRESSIONS)
    display(Image(filename=str(LOSS_PLOT)))
    display(Image(filename=str(DISTANCE_PLOT)))


## 6. Export the experiment log

The result tables are updated after every successful seed in section 3. The experiment manifest records exact settings, seed list, source-data checksums, and extensions. The log below is a machine-generated progress record, not a scientific conclusion.

In [ ]:
record = json.loads((EXPERIMENT_DIR / 'experiment.json').read_text())
completed, planned = int(MANIFEST.status.eq('complete').sum()), len(MANIFEST)
log = [
    '# Figure 2/3 seed-stability experiment', '',
    f'Experiment: `{EXPERIMENT_DIR.name}`',
    f'Completed fits: {completed}/{planned}.',
    f'Seeds: {SEEDS[0]}–{SEEDS[-1]} ({len(SEEDS)} per variant).',
    f'Scenarios: {SCENARIOS}; DESI error versions: {ERROR_VERSIONS}.',
    f'Fixed PySR search settings: {CONFIG["search"]}.', '',
    'The paper-table errors are a constructed sensitivity check, not the authors original covariance.',
    'Seed envelopes quantify search variation on fixed data, not measurement uncertainty.',
    'The discrepancy with printed paper losses remains a separate audit question.',
]
(EXPERIMENT_DIR / 'analysis_log.md').write_text('\n'.join(log) + '\n')
print(f'Saved {completed}/{planned} fits in {EXPERIMENT_DIR}')
print('Experiment status:', record['status'])


## 7. Interpretation checklist

After completion, ask whether one family recurs across seeds; whether its coefficients and $D_M/r_d$ curves are stable; whether low-loss expressions remain finite, positive for $z>0$, and consistent with $D_M(0)=0$; and whether the conclusions change between supernova datasets or DESI-error conventions. A single matching seed does not establish reproducibility. Record any manual setting changes in a **new** experiment rather than mixing them into this seed ensemble.